# Tokenizer 基础：文字怎么变数字

> **本 Part 目标**：搞清楚大模型读文字前发生了什么。我们会先看一个真实 tokenizer 把一句话切成了什么，再一步步手写字符级、词级和子词级 tokenizer。

你可以先把 tokenizer 想成“文字进模型前的翻译器”。

大模型不会直接读取字符串。你输入的文本，会先被 tokenizer 切成 token，再映射成 token ID。下面先跑一个真实 tokenizer，看模型入口到底长什么样。


In [1]:
# 先用真实 tokenizer 跑出结果
try:
    import tiktoken

    modern_tokenizer = tiktoken.get_encoding("gpt2")
    print("真实 tokenizer: GPT-2 byte-level BPE")
    print(f"词表大小: {modern_tokenizer.n_vocab}")
except Exception as e:
    modern_tokenizer = None
    print("未能加载 tiktoken。安装 tiktoken 后可运行真实 tokenizer 演示。")
    print(f"原因: {e}")


def inspect_modern_tokenizer(text):
    """打印真实 tokenizer 的 token、ID 和底层 bytes"""
    ids = modern_tokenizer.encode(text)
    tokens = [modern_tokenizer.decode([tok_id]) for tok_id in ids]

    print("=" * 68)
    print(f"原文: {text!r}")
    print(f"Token IDs: {ids}")
    print(f"token 数: {len(ids)}")
    print("逐个 token 查看:")
    for i, (tok_id, token) in enumerate(zip(ids, tokens)):
        visible = token.replace("\n", "\\n").replace("\t", "\\t")
        token_bytes = list(modern_tokenizer.decode_single_token_bytes(tok_id))
        print(f"  {i:02d} | id={tok_id:<6} | token={visible!r} | bytes={token_bytes}")


if modern_tokenizer is not None:
    examples = [
        "the cat sat on the mat",
        "Tokenizer 会把文字变成数字。",
    ]

    for example in examples:
        inspect_modern_tokenizer(example)

    print("=" * 68)
    print("关键观察：模型真正吃进去的是 token ID，不是原始字符串。")
    print("接下来我们从最简单的字符级 tokenizer 开始，把这件事手写出来。")


真实 tokenizer: GPT-2 byte-level BPE
词表大小: 50257
原文: 'the cat sat on the mat'
Token IDs: [1169, 3797, 3332, 319, 262, 2603]
token 数: 6
逐个 token 查看:
  00 | id=1169   | token='the' | bytes=[116, 104, 101]
  01 | id=3797   | token=' cat' | bytes=[32, 99, 97, 116]
  02 | id=3332   | token=' sat' | bytes=[32, 115, 97, 116]
  03 | id=319    | token=' on' | bytes=[32, 111, 110]
  04 | id=262    | token=' the' | bytes=[32, 116, 104, 101]
  05 | id=2603   | token=' mat' | bytes=[32, 109, 97, 116]
原文: 'Tokenizer 会把文字变成数字。'
Token IDs: [30642, 7509, 220, 27670, 248, 162, 232, 232, 23877, 229, 27764, 245, 20998, 246, 22755, 238, 46763, 108, 27764, 245, 16764]
token 数: 21
逐个 token 查看:
  00 | id=30642  | token='Token' | bytes=[84, 111, 107, 101, 110]
  01 | id=7509   | token='izer' | bytes=[105, 122, 101, 114]
  02 | id=220    | token=' ' | bytes=[32]
  03 | id=27670  | token='�' | bytes=[228, 188]
  04 | id=248    | token='�' | bytes=[154]
  05 | id=162    | token='�' | bytes=[230]
  06 | id=232    | 

## 学习地图

这一节你只需要抓住 4 件事：

1. **Tokenizer 是翻译器**：把文本变成 token ID，也能把 ID 拼回文本。
2. **字符级很稳**：不会轻易遇到生词，但序列会很长。
3. **词级很短**：token 少很多，但词表会爆炸，还会遇到 OOV。
4. **子词级折中**：常见片段合起来，少见词拆开，这就是下一节 BPE 的方向。


## 1. 先说清楚：Token 和 Tokenizer 是什么

先给一个一锤定音的定义：

**Token**：模型一次处理的最小文本单位。

它可以是一个字符、一个词，也可以是一个常见子词片段。

```text
字符级 token:  "cat" → ["c", "a", "t"]
词级 token:    "the cat" → ["the", "cat"]
子词级 token:  "playing" → ["play", "ing"]
```

**Tokenizer**：负责在“文本”和“token ID 序列”之间来回翻译的工具。

它做两件事：

```text
文本  --encode-->  token ID 序列
ID序列 --decode-->  文本
```

为什么一定要有 Tokenizer？

因为 LLM 里面的神经网络只会处理数字张量，不会直接处理字符串。你输入的 `"the cat"`，必须先被切成 token，再把每个 token 映射成一个 ID。

比如字符级切法可能是：

```text
"the cat" → ["t", "h", "e", " ", "c", "a", "t"]
          → [5, 12, 3, 0, 6, 1, 8]
```

注意：这里的数字只是编号，不是大小关系。`12` 不代表比 `3` 更厉害，它只是另一个 token 的身份证号。

所以这一节真正要回答的问题是：**一段文字到底应该怎么切成 token？**

下面我们从最小语料开始，试三种切法：字符级、词级、子词级。

顺手先认识一个现代 LLM 里很常见的东西：**special token**。

普通 token 来自文本本身，比如 `the`、`猫`、`ing`。
Special token 是人手动约定的控制符号，比如：

```text
<BOS>  表示序列开始
<EOS>  表示序列结束
<PAD>  表示这里是补齐出来的位置
```

它们也会有自己的 ID。区别只是：普通 token 表示文本内容，special token 表示“边界”和“规则”。
后面写 Mini-GPT 时，你还会看到 `<think>` 这种用来标记思考区间的 special token。


## 2. 准备语料

Tokenizer 需要先知道自己会遇到哪些文本，所以我们准备一小段语料。

这里先用英文，因为空格边界很明显，适合观察。等你看懂原理，中文只是切分规则更麻烦，不是另一个世界。


In [2]:
# 我们的语料库 — 故意选有重复模式的简单句子
# 这样 BPE 合并时能清楚看到高频 pair 如何被合并
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog",
    "i love my cat",
    "i love my dog",
    "the cat is cute",
    "the dog is happy",
    "the mat is soft",
    "the log is hard",
    "cats and dogs are friends",
]

print(f"语料库共 {len(corpus)} 条句子")
for i, s in enumerate(corpus):
    print(f"  [{i}] {s}")

total_chars = sum(len(s) for s in corpus)
print(f"\n总字符数: {total_chars}")
print(f"总词数（按空格算）: {sum(len(s.split()) for s in corpus)}")

语料库共 10 条句子
  [0] the cat sat on the mat
  [1] the dog sat on the log
  [2] the cat and the dog
  [3] i love my cat
  [4] i love my dog
  [5] the cat is cute
  [6] the dog is happy
  [7] the mat is soft
  [8] the log is hard
  [9] cats and dogs are friends

总字符数: 175
总词数（按空格算）: 46


**先看：计算机眼里的文字**

你在屏幕上看到 `the cat`，计算机底层看到的是字符编号。

```
文本:    t    h    e         c    a    t
        ↓    ↓    ↓    ↓    ↓    ↓    ↓
ASCII:  116  104  101  32  99   97   116
```

所以 Tokenizer 的真正问题不是“能不能变数字”，而是：**切多细？**

- 按字符切：每个字母一个 token
- 按词切：每个单词一个 token
- 按子词切：常见片段变 token，比如 `ing`、`the`、`cat`

我们一个个试。


## 3. 字符级

### 按字符切

字符级 Tokenizer 的想法最朴素：**每个字符都是一个 token。**

```
"the cat"
  ↓ 按字符切
['t', 'h', 'e', ' ', 'c', 'a', 't']
  ↓ 查表给编号
[5, 12, 3, 0, 6, 1, 8]
```

词表（vocab）就是一张“字符 → ID”的字典。

优点很明显：简单、稳定、几乎不怕生词。我们先手写一个看看。


In [3]:
class CharTokenizer:
    """
    字符级 Tokenizer：一个字符 = 一个 token
    只有两个核心方法：encode（编码）和 decode（解码）
    """
    
    def __init__(self):
        # stoi = string-to-id，把字符映射到数字
        # itos = id-to-string，把数字映射回字符
        self.stoi = {}
        self.itos = {}
    
    def train(self, texts):
        """
        训练：从语料中收集所有出现过的字符，建立词表
        
        步骤：
        1. 把所有文本拼成一个长字符串
        2. 用 set 去重 → 得到所有字符
        3. 排序 → 给每个字符分配一个唯一 ID
        """
        all_text = "".join(texts)
        chars = sorted(list(set(all_text)))
        
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for i, ch in enumerate(chars)}
        
        print(f"=== 字符级 Tokenizer 训练完成 ===")
        print(f"词表大小: {len(self.stoi)} 个 token")
        print(f"词表内容: {chars}")
    
    def encode(self, text):
        """编码：文本 → token ID 列表"""
        return [self.stoi[ch] for ch in text]
    
    def decode(self, ids):
        """解码：token ID 列表 → 文本"""
        return "".join([self.itos[i] for i in ids])


In [4]:
# 训练并测试
char_tokenizer = CharTokenizer()
char_tokenizer.train(corpus)

# 拿一句来试试
text = "the cat"
ids = char_tokenizer.encode(text)
recovered = char_tokenizer.decode(ids)

print(f"\n=== 编码/解码测试 ===")
print(f"原文:     '{text}'")
print(f"Token IDs: {ids}")
print(f"解码回来: '{recovered}'")
print(f"\n关键观察：原文 {len(text)} 个字符 → 产生 {len(ids)} 个 token → 一样多！")
print(f"每个字符都变成一个 token，没有压缩。")

=== 字符级 Tokenizer 训练完成 ===
词表大小: 20 个 token
词表内容: [' ', 'a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'y']

=== 编码/解码测试 ===
原文:     'the cat'
Token IDs: [16, 7, 4, 0, 2, 1, 16]
解码回来: 'the cat'

关键观察：原文 7 个字符 → 产生 7 个 token → 一样多！
每个字符都变成一个 token，没有压缩。


### 字符级的问题

字符级好懂，但它有两个麻烦。

**问题 1：序列太长。**

`"the cat"` 是 7 个字符，也就是 7 个 token。一篇 1000 字符的文章就是 1000 个 token。

LLM 里的 Self-Attention 对长度很敏感，长度变 10 倍，计算量可能接近 100 倍。

**问题 2：意思被打碎。**

`cat` 本来是一个完整概念，但字符级会拆成 `c`、`a`、`t`。模型还得自己重新学会这三个字符经常组成“猫”。

所以我们自然会想：那直接按词切是不是更好？


## 4. 词级

### 按词切

词级 Tokenizer 的想法是：**每个单词都是一个 token。**

```
"the cat sat on the mat"
  ↓ 按空格切
['the', 'cat', 'sat', 'on', 'the', 'mat']
  ↓ 查表
[0, 1, 2, 3, 0, 4]
```

优点是序列短多了，`the cat` 只需要 2 个 token。

但它有一个致命问题：**没见过的词怎么办？**

如果词表里没有 `cats`，模型就不知道该给它什么 ID，这就是 OOV（out of vocabulary）。


In [5]:
class WordTokenizer:
    """
    词级 Tokenizer：按空格切分，一个词 = 一个 token
    结构和 CharTokenizer 一模一样，只是切分粒度从「字符」变成了「词」
    """
    
    def __init__(self):
        self.stoi = {}
        self.itos = {}
    
    def train(self, texts):
        """从语料中收集所有出现过的词"""
        all_words = set()
        for text in texts:
            words = text.split()
            all_words.update(words)
        
        all_words = sorted(list(all_words))
        self.stoi = {w: i for i, w in enumerate(all_words)}
        self.itos = {i: w for i, w in enumerate(all_words)}
        
        print(f"=== 词级 Tokenizer 训练完成 ===")
        print(f"词表大小: {len(self.stoi)} 个词")
        print(f"词表内容: {all_words}")
    
    def encode(self, text):
        """把文本按空格切开，每个词查表"""
        return [self.stoi[w] for w in text.split()]
    
    def decode(self, ids):
        """把 ID 查表变回词，用空格拼起来"""
        return " ".join([self.itos[i] for i in ids])


In [6]:
# 训练并测试
word_tokenizer = WordTokenizer()
word_tokenizer.train(corpus)

text = "the cat sat on the mat"
ids = word_tokenizer.encode(text)
recovered = word_tokenizer.decode(ids)

print(f"\n=== 编码/解码测试 ===")
print(f"原文:     '{text}'")
print(f"Token IDs: {ids}")
print(f"解码回来: '{recovered}'")
print(f"\n关键观察：原文 {len(text.split())} 个词 → 产生 {len(ids)} 个 token → 序列比字符级短多了！")

=== 词级 Tokenizer 训练完成 ===
词表大小: 20 个词
词表内容: ['and', 'are', 'cat', 'cats', 'cute', 'dog', 'dogs', 'friends', 'happy', 'hard', 'i', 'is', 'log', 'love', 'mat', 'my', 'on', 'sat', 'soft', 'the']

=== 编码/解码测试 ===
原文:     'the cat sat on the mat'
Token IDs: [19, 2, 17, 16, 19, 14]
解码回来: 'the cat sat on the mat'

关键观察：原文 6 个词 → 产生 6 个 token → 序列比字符级短多了！


In [7]:
# 但词级 tokenizer 有一个致命问题：遇到生词（OOV）怎么办？
print("=== OOV（Out Of Vocabulary）问题演示 ===")
print(f"当前词表: {list(word_tokenizer.stoi.keys())}")
print()

try:
    test_text = "the elephant sat"
    print(f"尝试编码: '{test_text}'")
    ids = word_tokenizer.encode(test_text)
    print(f"结果: {ids}")
except KeyError as e:
    # 看看出错时发生了什么
    words = test_text.split()
    for w in words:
        if w in word_tokenizer.stoi:
            print(f"  ✅ '{w}' → ID {word_tokenizer.stoi[w]}  (词表里有)")
        else:
            print(f"  ❌ '{w}' → 不在词表中！报错！")
    print(f"\n结论：遇到生词 'elephant'，整个程序崩溃。")
    print(f"而在现实世界，你永远不可能预先把所有词都收进词表。")

=== OOV（Out Of Vocabulary）问题演示 ===
当前词表: ['and', 'are', 'cat', 'cats', 'cute', 'dog', 'dogs', 'friends', 'happy', 'hard', 'i', 'is', 'log', 'love', 'mat', 'my', 'on', 'sat', 'soft', 'the']

尝试编码: 'the elephant sat'
  ✅ 'the' → ID 19  (词表里有)
  ❌ 'elephant' → 不在词表中！报错！
  ✅ 'sat' → ID 17  (词表里有)

结论：遇到生词 'elephant'，整个程序崩溃。
而在现实世界，你永远不可能预先把所有词都收进词表。


## 5. 子词级

现在把三种思路放在一起看：

| 方案 | 词表大小 | 序列长度 | OOV 问题 | 直觉 |
|:---|:---|:---|:---|:---|
| 字符级 | 很小 | 很长 | 几乎没有 | 稳，但太碎 |
| 词级 | 很大 | 很短 | 容易出现 | 清楚，但太死 |
| 子词级 | 可控 | 中等 | 很少 | 折中 |

子词级想要的是：

```
"the cat sat"      → [the, cat, sat]
"cats and dogs"    → [cat, s, and, dog, s]
"unbelievable"     → [un, believ, able]
```

常见词整体保留，少见词拆成小片段。这样词表不会爆炸，遇到新词也不容易崩。


### BPE 直觉

BPE 的核心动作很简单：**找最常出现的相邻字符对，把它们合并成一个新 token。**

```
初始：每个字符一个 token
['t', 'h', 'e', ' ', 'c', 'a', 't', ...]
        ↓ 统计相邻 pair
('t', 'h') 出现很多次
        ↓ 合并
新 token: 'th'
        ↓ 继续统计，继续合并
```

重复很多次，词表就会从字符慢慢长出 `th`、`the`、`ing` 这样的子词。

下一节我们会从零手写 BPE，并把每一步合并都打印出来。


---

## 小结

确认你已经懂了这些：

1. ✅ Tokenizer 是文本和 token ID 之间的翻译器
2. ✅ encode 是文本到 ID，decode 是 ID 到文本
3. ✅ vocab 是所有合法 token 的集合，每个 token 有唯一 ID
4. ✅ 字符级简单稳定，但序列太长、意思太碎
5. ✅ 词级序列短，但词表大、遇生词容易 OOV
6. ✅ 子词级是折中：高频组合合并，低频词拆开

下一步：手写 BPE，看词表怎么一步步“长出来”。


---

## 本章作业：Tokenizer 基础

这 3 题分成两类：

1. **核心必须记住**：Tokenizer 的本质是 `文本 ↔ token ID`。
2. **现代 LLM 怎么用**：真实模型常用 special tokens、attention mask 和固定长度 batch。

> **使用 AI 的注意事项**：可以让 AI 给你提示、讲思路、检查每一步，但不要直接说“做完这道题”。骗自己不好玩，真正重要的是你亲手把文本变成 ID。

### 作业 1：核心机制 —— `encode` 到底做了什么

把 token 查表变成 ID，这是 Tokenizer 最核心的动作。

**小提示**：`stoi[token]` 就是把一个 token 变成它的身份证号。

In [ ]:
# 作业 1：核心机制 encode 填空
stoi = {"the": 0, "cat": 1, "sat": 2}
tokens = ["the", "cat", "sat"]

# TODO：把下面三引号里的内容替换成你的代码
ids = """在这里把 tokens 里的每个 token 转成 ID"""

assert not isinstance(ids, str), "请先替换三引号里的占位内容"
assert ids == [0, 1, 2], ids
print("✅ 作业 1 通过：你记住了 encode 的核心就是 token → ID")

### 作业 2：现代用法 —— 加上 special tokens

现代 LLM 不只看普通文本 token，还会用 `<BOS>` 表示开始、`<EOS>` 表示结束。

**小提示**：很多模型的输入会长得像 `[BOS] + 正文 tokens + [EOS]`。

In [ ]:
# 作业 2：special tokens 填空
stoi = {"<BOS>": 0, "<EOS>": 1, "the": 2, "cat": 3}
tokens = ["the", "cat"]

# TODO：把下面三引号里的内容替换成你的代码
input_ids = """在这里给正文前后加上 <BOS> 和 <EOS> 的 ID"""

assert not isinstance(input_ids, str), "请先替换三引号里的占位内容"
assert input_ids == [0, 2, 3, 1], input_ids
print("✅ 作业 2 通过：你知道现代 LLM 输入里常有 special tokens")

### 作业 3：现代用法 —— padding 和 attention mask

一个 batch 里句子长短不同，通常会 padding 到同样长度，并用 attention mask 标出哪些位置是真 token。

**小提示**：真实 token 的 mask 是 `1`，padding 的 mask 是 `0`。

In [ ]:
# 作业 3：padding + attention mask 填空
PAD_ID = 0
ids = [5, 6, 7]
max_len = 5

# TODO：把下面三引号里的内容替换成你的代码
padded_ids = """在这里把 ids 补到 max_len 长度"""
attention_mask = """在这里标出真实 token 和 PAD 位置"""

assert not isinstance(padded_ids, str), "请先替换 padded_ids 的占位内容"
assert not isinstance(attention_mask, str), "请先替换 attention_mask 的占位内容"
assert padded_ids == [5, 6, 7, 0, 0], padded_ids
assert attention_mask == [1, 1, 1, 0, 0], attention_mask
print("✅ 作业 3 通过：你理解了现代 tokenizer 输出里常见的 attention_mask")